# Численное моделирование обтекания цилиндра дозвуковым потоком
В данной работе решается задача обтекания круглого цилиндра идеальным сжимаемым газом с использованием потенциальной постановки и метода SLOR.

## Математическая модель
Уравнение неразрывности для потенциала скорости $\phi$ ($\vec{v} = \nabla \phi$):
$$\frac{\partial}{\partial x} (\rho \phi_x) + \frac{\partial}{\partial y} (\rho \phi_y) = 0$$

В криволинейных координатах $(\xi, \eta)$, связанных с декартовыми как $x = e^\xi \cos \eta, y = e^\xi \sin \eta$:
$$\frac{\partial}{\partial \xi} (\rho \phi_\xi) + \frac{\partial}{\partial \eta} (\rho \phi_\eta) = 0$$

### Физические параметры
- Плотность: $$\rho = [1 + \frac{\gamma-1}{2} M_\infty^2 (1 - q^2)]^{\frac{1}{\gamma-1}}$$
- Квадрат скорости: $$q^2 = e^{-2\xi} [\phi_\xi^2 + \phi_\eta^2]$$ (обезразмерено на $U_\infty$)
- Локальное число Маха: $$M_{loc} = q M_\infty/ a$$, где $$a^2 = 1 + \frac{\gamma-1}{2}M_\infty^2(1 - q^2)$$
- Коэффициент давления: $$C_p = \frac{2}{\gamma M_\infty^2} (\rho^\gamma - 1)$$






In [71]:
import matplotlib.pyplot as plt
import numpy as np
# Параметры задачи
gamma = 1.4
xi_max = 3.0
N_xi = 100
N_eta = 120
h_xi = xi_max / N_xi
h_eta = 2 * np.pi / N_eta  # Область от 0 до 2*pi

eps = 1e-5
omega = 1.5

xi = np.linspace(0, xi_max, N_xi + 1)
eta = np.linspace(0, 2 * np.pi, N_eta + 1)
XI, ETA = np.meshgrid(xi, eta, indexing='ij')
J_xi = np.exp(-2*XI)



## Граничные условия

### 1. Поверхность цилиндра ($\xi=0$)
Условие непротекания: $\phi_{-1, j} = \phi_{1, j}$. В методе SLOR это реализуется путем модификации коэффициента связи в первом узле: $C_0 = A_0 + C_0$.

### 2. Внешняя граница ($\xi=\xi_{max}$)
Условие Дирихле: $\phi_{N_\xi, j} = e^{\xi_{max}} \cos \eta_j$.

### 3. Симметрия на осях ($\eta=0, 2\pi$)
Хотя расчетная сетка охватывает $2\pi$, на границах $\eta=0$ и $\eta=2\pi$ ставятся условия симметрии $\phi_{i,-1} = \phi_{i,1}$ и $\phi_{i,N_\eta+1} = \phi_{i,N_\eta-1}$. 

Это реализуется через удвоение влияния соседа в крайних строках:
- Для $j=0$: Правая часть $D$ содержит удвоенный вклад от строки $j=1$.
- Для $j=N_\eta$: Правая часть $D$ содержит удвоенный вклад от строки $j=N_\eta-1$.


## Метод SLOR
Для каждой линии $\eta = \text{const}$ решается трехдиагональная система:
$$A_i \phi_{i-1, j}^* + B_i \phi_{i, j}^* + C_i \phi_{i+1, j}^* = D_i$$
Затем применяется релаксация:
$$\phi_{i, j}^{k+1} = \phi_{i, j}^k + \omega (\phi_{i, j}^* - \phi_{i, j}^k)$$

### Коэффициенты системы:
Аппроксимация уравнения $\frac{\partial}{\partial \xi} (\rho \phi_\xi) + \frac{\partial}{\partial \eta} (\rho \phi_\eta) = 0$:

$$A_i = \frac{\rho_{i-1/2, j}}{h_\xi^2}$$
$$C_i = \frac{\rho_{i+1/2, j}}{h_\xi^2}$$
$$B_i = -\left( \frac{\rho_{i+1/2, j} + \rho_{i-1/2, j}}{h_\xi^2} + \frac{\rho_{i, j+1/2} + \rho_{i, j-1/2}}{h_\eta^2} \right)$$
$$D_i = -\frac{\rho_{i, j+1/2} \phi_{i, j+1}^k + \rho_{i, j-1/2} \phi_{i, j-1}^{k+1}}{h_\eta^2}$$

Где $\rho_{i \pm 1/2, j} = \frac{\rho_{i, j} + \rho_{i \pm 1, j}}{2}$ — полуцелые значения плотности.


### Метод прогонки для (A,B,C, D) -> result

In [72]:
from numba import njit
@njit
def solve_tridiagonal(A, B, C, D):
    n = len(D)
    c_prime = np.zeros(n) ; d_prime = np.zeros(n) 
    c_prime[0] = C[0] / B[0] ; d_prime[0] = D[0] / B[0]   
    for i in range(1, n):
        denom = B[i] - A[i] * c_prime[i-1]
        c_prime[i] = C[i] / denom
        d_prime[i] = (D[i] - A[i] * d_prime[i-1]) / denom       
    x = np.zeros(n)
    x[-1] = d_prime[-1]
    for i in range(n-2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i+1]
    return x

### Обновляем плотность


In [73]:
def get_density(phi, M_inf):
    if M_inf == 0:
        return np.ones_like(phi)
    
    # empty_like быстрее, так как не обнуляет память
    phi_xi = np.empty_like(phi)
    phi_eta = np.empty_like(phi)
    
    # Центральные разности
    phi_xi[1:-1, :] = (phi[2:, :] - phi[:-2, :]) / (2 * h_xi)
    phi_xi[0, :] = 0 
    phi_xi[N_xi, :] = (phi[N_xi, :] - phi[N_xi-1, :]) / h_xi
    
    # Границы по eta (симметрия)
    phi_eta[:, 0] = 0
    phi_eta[:, N_eta] = 0
    phi_eta[:, 1:N_eta] = (phi[:, 2:N_eta+1] - phi[:, 0:N_eta-1]) / (2 * h_eta)
    
    # Коэффициент метрики (J_xi или exp(-2*xi))
    #J_xi = np.exp(2 * XI) 
    q2 = J_xi * (phi_xi**2 + phi_eta**2)
    
    # Обязательно ограничиваем q2 для стабильности
    #q2 = np.clip(q2, 0, 1.0 + 2.0 / ((gamma - 1) * M_inf**2) - 0.01)
    
    Rho = (1 + 0.5 * (gamma - 1) * M_inf**2 * (1 - q2))**(1.0/(gamma-1))
    Rho[N_xi, :] = 1.0 # Фиксируем плотность на бесконечности

    return Rho


## Граничные условия

### 1. Поверхность цилиндра ($\xi=0$)
Условие непротекания: $\phi_{-1, j} = \phi_{1, j}$. В методе SLOR это реализуется путем модификации коэффициента связи в первом узле: $C_0 => 2*C_0$.

### 2. Внешняя граница ($\xi=\xi_{max}$)
Условие Дирихле: $\phi_{N_\xi, j} = e^{\xi_{max}} \cos \eta_j$.

### 3. Симметрия на осях ($\eta=0, 2\pi$)
Хотя расчетная сетка охватывает $2\pi$, на границах $\eta=0$ и $\eta=2\pi$ ставятся условия симметрии $\phi_{i,-1} = \phi_{i,1}$ и $\phi_{i,N_\eta+1} = \phi_{i,N_\eta-1}$. 

Это реализуется через удвоение влияния соседа в крайних строках:
- Для $j=0$: Правая часть $D$ содержит удвоенный вклад от строки $j=1$.
- Для $j=N_\eta$: Правая часть $D$ содержит удвоенный вклад от строки $j=N_\eta-1$.


## Решатель по столбцам

In [74]:
def slor_iteration_angular(phi, M_inf,rho, omega):
    phi_new = phi.copy()
    m = N_eta + 1  # Количество узлов по углу
    h_xi2_inv = 1.0 / (h_xi**2)
    h_eta2_inv = 1.0 / (h_eta**2)
    
    j_range = np.arange(m)
    A = np.zeros(m)
    C = np.zeros(m)
    # Итерируемся по радиальным уровням i (от поверхности до границы)
    for i in range(N_xi):
        # 1. Плотности по углу j (rho_{i, j+1/2})
        rho_jp = 0.5 * (rho[i, 0:m-1] + rho[i, 1:m])
        
        # 2. Формируем A_j и C_j для прогонки по углу
        
        C[:-1] = -rho_jp * h_eta2_inv
        A[1:] = -rho_jp * h_eta2_inv
        
        # 3. Вклады по радиальному направлению xi (фиксированное i)
        rho_ip = 0.5 * (rho[i, j_range] + rho[i+1, j_range]) # rho_{i+1/2, j}
        
        if i == 0:
            # Поверхность цилиндра (Нейман: phi_{-1} = phi_1)
            # Удвоенный вклад от соседа i=1
            B_xi = 2 * rho_ip * h_xi2_inv
            D = (2 * rho_ip * phi[1, j_range]) * h_xi2_inv
        else:
            # Внутренние радиусы
            rho_im = 0.5 * (rho[i, j_range] + rho[i-1, j_range]) # rho_{i-1/2, j}
            B_xi = (rho_ip + rho_im) * h_xi2_inv
            # Используем phi_new для уже обновленного i-1 и phi для i+1
            D = (rho_ip * phi[i+1, j_range] + rho_im * phi_new[i-1, j_range]) * h_xi2_inv
            
        # 4. Формируем центральный коэффициент B_j
        sum_rho_eta = np.zeros(m)
        sum_rho_eta[1:-1] = rho_jp[1:] + rho_jp[:-1]
        sum_rho_eta[0] = 2 * rho_jp[0]     # Симметрия на линии j=0
        sum_rho_eta[-1] = 2 * rho_jp[-1]   # Симметрия на линии j=N_eta
        
        B = sum_rho_eta * h_eta2_inv + B_xi
        
        # 5. Граничные условия по углу (линии симметрии j=0 и j=N_eta)
        C[0] = C[0] + (-rho_jp[0] * h_eta2_inv)
        A[0] = 0
        
        A[-1] = A[-1] + (-rho_jp[-1] * h_eta2_inv)
        C[-1] = 0

        # Решение системы для «кольца» i
        phi_star = solve_tridiagonal(A, B, C, D)
        phi_new[i, :] = phi[i, :] + omega * (phi_star - phi[i, :])
    
    return phi_new


## Решатель по строкам

In [75]:
def slor_iteration(phi, M_inf,rho, omega):
    
    phi_new = phi.copy()
    n = N_xi
    h_xi2_inv = 1.0 / (h_xi**2)
    h_eta2_inv = 1.0 / (h_eta**2)
    
    i_range = np.arange(n)
    ip1 = i_range + 1
    rho_im = np.zeros(n)
    for j in range(N_eta + 1):
        # 1. Плотности на гранях ячеек
        rho_ip = 0.5 * (rho[0:n, j] + rho[1:n+1, j])  # rho_{i+1/2, j}
        
        rho_im[1:] = rho_ip[:-1]                      # rho_{i-1/2, j}
        rho_im[0] = rho_ip[0]                         # Симметрия на стенке (i=0)
        
        # 2. Формируем коэффициенты A и C (отрицательные для -Laplace)
        A = -rho_im * h_xi2_inv
        C = -rho_ip * h_xi2_inv
        
        # 3. Формируем B и D в зависимости от j (все коэффициенты положительные)
        if j == 0:
            # Нижняя линия симметрии (удвоенный поток по eta)
            rho_jp = 0.5 * (rho[i_range, 0] + rho[i_range, 1])
            B = ( (rho_im + rho_ip) * h_xi2_inv + 2 * rho_jp * h_eta2_inv )
            D = (2 * rho_jp * phi[i_range, 1]) * h_eta2_inv
            
        elif j == N_eta:
            # Верхняя линия симметрии (удвоенный поток по eta)
            rho_jm = 0.5 * (rho[i_range, N_eta] + rho[i_range, N_eta-1])
            B = ( (rho_im + rho_ip) * h_xi2_inv + 2 * rho_jm * h_eta2_inv )
            D = (2 * rho_jm * phi_new[i_range, N_eta-1]) * h_eta2_inv
            
        else:
            # Внутренние строки
            rho_jp = 0.5 * (rho[i_range, j] + rho[i_range, j+1])
            rho_jm = 0.5 * (rho[i_range, j] + rho[i_range, j-1])
            B = ( (rho_im + rho_ip) * h_xi2_inv + (rho_jp + rho_jm) * h_eta2_inv )
            D = (rho_jp * phi[i_range, j+1] + rho_jm * phi_new[i_range, j-1]) * h_eta2_inv
        
        # 4. Граничное условие на поверхности (i=0): Нейман
        C[0] = C[0] + A[0]
        A[0] = 0
        
        # 5. Граничное условие на бесконечности (i=N_xi-1): Дирихле
        D[-1] -= C[-1] * phi[N_xi, j]
        #B[-1] = 1
        C[-1] = 0
        #D[-1] = np.exp(-xi_max)*np.cos(j*np.pi/180)

        # Решение трехдиагональной системы
        phi_star = solve_tridiagonal(A, B, C, D)
        
        # Релаксация
        phi_new[0:n, j] = phi[0:n, j] + omega * (phi_star - phi[0:n, j])
    
    return phi_new

## Итерационный процесс

In [76]:
def solve_for_mach(M_inf, phi_init=None):
    # Используем улучшенное начальное приближение для ускорения сходимости
    if phi_init is None:
        phi = (np.exp(XI) + np.exp(-XI)) * np.cos(ETA)
    else:
        phi = (np.exp(XI) + np.exp(-XI)) * np.cos(ETA)
        #phi = phi_init.copy()

    # ПРИНУДИТЕЛЬНО задаем правую границу (i = N_xi)
    # Это соответствует потенциалу набегающего потока на бесконечности
    #phi[-1, :] = np.exp(XI[-1, :]) * np.cos(ETA[-1, :])
    rho = get_density(phi, M_inf)
    rho[:] = 1.0
    for it in range(2000):
        
        phi_prev = phi.copy()
        #if it %2 == 0:
        
        phi = slor_iteration(phi, M_inf, rho, omega)
        #if it %10 ==0:
        #rho = get_density(phi, M_inf)

        
        if it % 10 == 0:
            err = np.max(np.abs(phi - phi_prev))
            if err < eps:
                print(f"M={M_inf} сошлось за {it} итераций (err={err:.2e})")
                break
        
        if it == 1999:    
            print(f"M={M_inf} несошлось за {it} итераций (err={err:.2e})")
                  

        if np.any(np.isnan(phi)):
            print(f"M={M_inf} РАСХОДИМОСТЬ на итерации {it}")
            return phi_prev
            
    return phi

## Запуск расчета и визуализация

In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

machs = [0., 0.1, 0.2, 0.3, 0.4]
results = {}
phi_current = None

for M in machs:
    phi_current = solve_for_mach(M, phi_current)
    results[M] = phi_current

table_data = []
idx_90 = np.argmin(np.abs(eta - np.pi/2))

for M in machs:
    phi = results[M]
    # Характеристики на цилиндре (i=0)
    phi_eta = np.zeros(N_eta + 1)
    # Внутренние узлы по eta
    phi_eta[1:N_eta] = (phi[0, 2:N_eta+1] - phi[0, 0:N_eta-1]) / (2 * h_eta)
    # Границы симметрии: производная по нормали к оси (phi_eta) равна 0
    phi_eta[0] = 0
    phi_eta[N_eta] = 0
    
    q2 = phi_eta**2 # на цилиндре xi=0, phi_xi=0
    q = np.sqrt(q2)
    
    if M == 0:
        cp = 1 - q2
        m_loc = np.zeros_like(q2)
    else:
        rho = get_density(phi, 0)
        rho[:] = 1.0
        #rho = (1 + 0.5 * (gamma - 1) * M**2 * (1 - q2))**(1/(gamma-1))
        cp = 2 / (gamma * M**2) * (rho**gamma - 1)
        a2 = 1 + 0.5 * (gamma - 1) * M**2 * (1 - q2)
        m_loc = q * M / np.sqrt(a2)
    
    # Сохранение полной таблицы (добавлено q)
    df_m = pd.DataFrame({
        'eta_deg': eta * 180 / np.pi,
        'q': q,
        'Cp': cp,
        'M_loc': m_loc
    })
    df_m.to_csv(f'results_slor_M_{M}.csv', index=False)
    
    # Данные для сводки (в точке pi/2) (добавлено q_pi/2)
    table_data.append({
        'M_inf': M, 
        'q_pi/2': q[idx_90],
        'Cp_pi/2': cp[idx_90], 
        'M_loc_pi/2': m_loc[idx_90]
    })

df_results = pd.DataFrame(table_data)
print("\nРезультаты (SLOR) в точке pi/2:")
print(df_results.to_string(index=False))

# Визуализация распределений Cp и M_loc
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
for M in machs:
    df_m = pd.read_csv(f'results_slor_M_{M}.csv')
    plt.plot(df_m['eta_deg'], df_m['Cp'], label=f'M={M}')
plt.title('Коэффициент давления Cp (SLOR)')
plt.xlabel('eta (deg)'); plt.ylabel('Cp'); plt.legend(); plt.grid()

plt.subplot(1, 2, 2)
for M in machs:
    if M == 0: continue
    df_m = pd.read_csv(f'results_slor_M_{M}.csv')
    plt.plot(df_m['eta_deg'], df_m['M_loc'], label=f'M={M}')
plt.title('Локальное число Маха M_loc (SLOR)')
plt.xlabel('eta (deg)'); plt.ylabel('M_loc'); plt.legend(); plt.grid()
plt.show()

# Новый график: q(M_inf) в точке pi/2
plt.figure(figsize=(8, 5))
plt.plot(df_results['M_inf'], df_results['q_pi/2'], 'o-r', lw=2)
plt.title('Зависимость скорости q в точке pi/2 от числа Маха M_inf')
plt.xlabel('M_inf'); plt.ylabel('q (at pi/2)'); plt.grid()
plt.show()


M=0.0 сошлось за 50 итераций (err=9.01e-06)
M=0.1 сошлось за 50 итераций (err=9.01e-06)
M=0.2 сошлось за 50 итераций (err=9.01e-06)
M=0.3 сошлось за 50 итераций (err=9.01e-06)
M=0.4 сошлось за 50 итераций (err=9.01e-06)


ValueError: Per-column arrays must each be 1-dimensional